In [1]:
import pandas as pd
import numpy as np
import joblib

from sqlalchemy import create_engine, URL, text
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors

In [2]:
connection_url = URL.create(
    drivername="mysql+pymysql",
    username="root",
    password="12345",
    host="localhost",
    port=3306,
    database="tmdb_movie_project"
)

engine = create_engine(connection_url)

In [3]:
with engine.connect() as connection:
    print(connection.execute(text("SELECT DATABASE()")).scalar())

tmdb_movie_project


In [4]:
movies = pd.read_sql("select * from movies", engine)
movie_genres = pd.read_sql("select * from movie_genres", engine)
movie_keywords = pd.read_sql("select * from movie_keywords", engine)
movie_writers = pd.read_sql("select * from movie_writers", engine)
movie_cast = pd.read_sql("select * from movie_cast", engine)

In [5]:
movies.shape, movie_genres.shape, movie_keywords.shape, movie_writers.shape, movie_cast.shape

((6764, 15), (18159, 2), (85720, 2), (4635, 2), (59674, 4))

In [6]:
# combine the genres, keywords, writers and cast for each movie

genres_text = movie_genres.groupby("movie_id")["genre"].agg(" ".join).reset_index(name="genres_text")

keywords_text = movie_keywords.groupby("movie_id")["keyword"].agg(" ".join).reset_index(name="keywords_text")

writers_text = movie_writers.groupby("movie_id")["writer"].agg(" ".join).reset_index(name="writers_text")

cast_text = movie_cast.groupby("movie_id")["actor"].agg(" ".join).reset_index(name="cast_text")


# create a separate dataframe for the recommendation system

movies_ml = movies.copy()

movies_ml = movies_ml.merge(genres_text, on="movie_id", how="left")
movies_ml = movies_ml.merge(keywords_text, on="movie_id", how="left")
movies_ml = movies_ml.merge(writers_text, on="movie_id", how="left")
movies_ml = movies_ml.merge(cast_text, on="movie_id", how="left")


# handle missing text

text_columns = [
    "overview",
    "director",
    "genres_text",
    "keywords_text",
    "writers_text",
    "cast_text"
]

movies_ml[text_columns] = movies_ml[text_columns].fillna("")

In [7]:
print(movies_ml.shape)

movies_ml[
    ["title", "genres_text", "keywords_text", "writers_text", "cast_text"]
].head()

(6764, 19)


,title,genres_text,keywords_text,writers_text,cast_text
0,Four Rooms,Comedy,anthology bet hoodlum hotel hotel room los ang...,Alexandre Rockwell Allison Anders Quentin Tara...,Tim Roth Jennifer Beals David Proval Ione Skye...
1,Star Wars,Action Adventure Science Fiction,android desert empire freedom galaxy hermit op...,George Lucas,Mark Hamill Harrison Ford Carrie Fisher Peter ...
2,Finding Nemo,Adventure Animation Family,3d animation aftercreditsstinger anthropomorph...,,Albert Brooks Ellen DeGeneres Alexander Gould ...
3,Forrest Gump,Comedy Drama Romance,1960s 1970s alabama america anti war protest a...,,Tom Hanks Robin Wright Gary Sinise Sally Field...
4,American Beauty,Drama,adultery age difference camcorder cheating che...,Alan Ball,Kevin Spacey Annette Bening Thora Birch Wes Be...


In [8]:
movies_ml["combined_features"] = (
    movies_ml["genres_text"] + " " +
    movies_ml["overview"] + " " +
    movies_ml["keywords_text"] + " " +
    movies_ml["cast_text"] + " " +
    movies_ml["director"] + " " +
    movies_ml["writers_text"]
).str.lower().str.strip()

In [9]:
movies_ml[["title", "combined_features"]].head(3)

,title,combined_features
0,Four Rooms,comedy it's ted the bellhop's first night on t...
1,Star Wars,action adventure science fiction princess leia...
2,Finding Nemo,"adventure animation family nemo, an adventurou..."


In [10]:
text_columns = [
    "overview",
    "director",
    "genres_text",
    "keywords_text",
    "writers_text",
    "cast_text",
    "combined_features"
]

movies_ml[text_columns].isna().sum()

overview             0
director             0
genres_text          0
keywords_text        0
writers_text         0
cast_text            0
combined_features    0
dtype: int64

In [11]:
(movies_ml["combined_features"].str.strip() == "").sum()

np.int64(1)

In [12]:
empty_features = movies_ml["combined_features"].str.strip() == ""

movies_ml.loc[empty_features, "combined_features"] = (
    movies_ml.loc[empty_features, "title"].fillna("").str.lower()
)

In [13]:
(movies_ml["combined_features"].str.strip() == "").sum()

np.int64(0)

In [14]:
def recommend_movies(title, n=10):
    matches = np.flatnonzero(
        movies_ml["title"].str.lower().eq(title.lower()).to_numpy()
    )

    if len(matches) == 0:
        print("Movie not found.")
        return

    movie_index = matches[0]

    distances, indices = knn_model.kneighbors(
        tfidf_matrix[movie_index],
        n_neighbors=min(n + 1, len(movies_ml))
    )

    recommended_indices = indices[0][1:]
    similarity_scores = 1 - distances[0][1:]

    recommendations = movies_ml.iloc[recommended_indices][
        [
            "title",
            "genres_text",
            "overview",
            "user_score",
            "runtime_minutes",
            "poster_url",
            "movie_url"
        ]
    ].copy()

    recommendations["similarity_score"] = similarity_scores

    return recommendations.reset_index(drop=True)

In [15]:
tfidf_vectorizer = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2),
    min_df=1
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    movies_ml["combined_features"]
)

knn_model = NearestNeighbors(
    metric="cosine",
    algorithm="brute",
    n_neighbors=11
)

knn_model.fit(tfidf_matrix)

,"n_neighbors n_neighbors: int, default=5Number of neighbors to use by default for :meth:`kneighbors` queries.",11
,"algorithm algorithm: {'auto', 'ball_tree', 'kd_tree', 'brute'}, default='auto'Algorithm used to compute the nearest neighbors:- 'ball_tree' will use :class:`BallTree`- 'kd_tree' will use :class:`KDTree`- 'brute' will use a brute-force search.- 'auto' will attempt to decide the most appropriate algorithm based on the values passed to :meth:`fit` method.Note: fitting on sparse input will override the setting ofthis parameter, using brute force.",'brute'
,"metric metric: str or callable, default='minkowski'Metric to use for distance computation. Default is ""minkowski"", whichresults in the standard Euclidean distance when p = 2. See thedocumentation of `scipy.spatial.distance<https://docs.scipy.org/doc/scipy/reference/spatial.distance.html>`_ andthe metrics listed in:class:`~sklearn.metrics.pairwise.distance_metrics` for valid metricvalues.If metric is ""precomputed"", X is assumed to be a distance matrix andmust be square during fit. X may be a :term:`sparse graph`, in whichcase only ""nonzero"" elements may be considered neighbors.If metric is a callable function, it takes two arrays representing 1Dvectors as inputs and must return one value indicating the distancebetween those vectors. This works for Scipy's metrics, but is lessefficient than passing the metric name as a string.",'cosine'
,"radius radius: float, default=1.0Range of parameter space to use by default for :meth:`radius_neighbors`queries.",1.0
,"leaf_size leaf_size: int, default=30Leaf size passed to BallTree or KDTree. This can affect thespeed of the construction and query, as well as the memoryrequired to store the tree. The optimal value depends on thenature of the problem.",30
,"p p: float (positive), default=2Parameter for the Minkowski metric fromsklearn.metrics.pairwise.pairwise_distances. When p = 1, this isequivalent to using manhattan_distance (l1), and euclidean_distance(l2) for p = 2. For arbitrary p, minkowski_distance (l_p) is used.",2
,"metric_params metric_params: dict, default=NoneAdditional keyword arguments for the metric function.",None
,"n_jobs n_jobs: int, default=NoneThe number of parallel jobs to run for neighbors search.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details.",None
Name,Type,Value
effective_metric_ effective_metric_: strMetric used to compute distances to neighbors.,str,'cosine'
effective_metric_params_ effective_metric_params_: dictParameters for the metric used to compute distances to neighbors.,dict,{}


In [16]:
recommend_movies("Finding Nemo")

,title,genres_text,overview,user_score,runtime_minutes,poster_url,movie_url,similarity_score
0,Finding Dory,Adventure Animation Family,Dory is reunited with her friends Nemo and Mar...,7.044,97.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/127380-findin...,0.261257
1,Shark Bait,Animation Family,"After losing everything, a young fish, Pi, goe...",4.948,77.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/19595-shark-bait,0.211963
2,Help! I'm a Fish,Adventure Animation Comedy Drama Family,Three children are accidentally transformed in...,6.700,80.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/12609-hjaelp-...,0.205103
3,Shark Tale,Action Animation Comedy Family,Oscar is a small fish whose big aspirations of...,6.046,90.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/10555-shark-tale,0.177554
4,Minions,Adventure Animation Comedy Family,"Minions Stuart, Kevin and Bob are recruited by...",6.426,91.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/211672-minions,0.168686
5,Mr. Peabody & Sherman,Adventure Animation Comedy Family Science Fiction,"A young boy and his dog, who happens to have a...",6.735,92.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/82703-mr-peab...,0.149623
6,Bambi II,Animation Drama Family,Return to the forest and join Bambi as he reun...,6.533,72.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/13205-bambi-ii,0.146397
7,The Life Aquatic with Steve Zissou,Adventure Comedy Drama,Renowned oceanographer Steve Zissou has sworn ...,7.072,119.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/421-the-life-...,0.142940
8,A Turtle's Tale 2: Sammy's Escape from Paradise,Adventure Animation Family,When Sammy and Ray find themselves trapped in ...,5.800,92.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/113594-a-turt...,0.138812
9,Over the Hedge,Adventure Animation Comedy Family,A scheming raccoon fools a mismatched family o...,6.609,83.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/7518-over-the...,0.126616


In [17]:
recommend_movies("Four Rooms")

,title,genres_text,overview,user_score,runtime_minutes,poster_url,movie_url,similarity_score
0,Pulp Fiction,Comedy Crime Thriller,"A burger-loving hit man, his philosophical par...",8.480,154.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/680-pulp-fiction,0.393757
1,Kill Bill: Vol. 1,Action Crime,"An assassin is shot by her ruthless employer, ...",7.970,111.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/24-kill-bill-...,0.289529
2,Once Upon a Time... in Hollywood,Comedy Drama Thriller,"Los Angeles, 1969. TV star Rick Dalton, a stru...",7.419,162.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/466272-once-u...,0.253816
3,Reservoir Dogs,Crime Thriller,A botched robbery indicates a police informant...,8.110,99.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/500-reservoir...,0.251005
4,Inglourious Basterds,Drama Thriller War,"In Nazi-occupied France during World War II, a...",8.217,153.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/16869-inglour...,0.250661
5,Django Unchained,Drama Western,"With the help of a German bounty hunter, a fre...",8.193,165.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/68718-django-...,0.240543
6,Kill Bill: The Whole Bloody Affair,Action Crime Drama Thriller,"A former assassin, known simply as The Bride, ...",8.114,254.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/414419-kill-b...,0.224632
7,Desperado,Action Crime Thriller,El Mariachi plunges headfirst into the dark bo...,6.930,104.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/8068-desperado,0.221592
8,The Hateful Eight,Drama Mystery Western,Bounty hunters seek shelter from a raging bliz...,7.756,188.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/273248-the-ha...,0.207061
9,Jackie Brown,Crime Drama Thriller,Jackie Brown is a flight attendant who gets ca...,7.358,154.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/184-jackie-brown,0.180280


In [18]:
recommend_movies("Star Wars")

,title,genres_text,overview,user_score,runtime_minutes,poster_url,movie_url,similarity_score
0,The Empire Strikes Back,Action Adventure Science Fiction,"The epic saga continues as Luke Skywalker, in ...",8.393,124.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/1891-the-empi...,0.461930
1,Return of the Jedi,Action Adventure Science Fiction,Luke Skywalker leads a mission to rescue his f...,7.900,132.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/1892-return-o...,0.390589
2,Star Wars: The Force Awakens,Action Adventure Science Fiction,Thirty years after defeating the Galactic Empi...,7.241,136.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/140607-star-w...,0.310756
3,Star Wars: Episode I - The Phantom Menace,Action Adventure Science Fiction,"Anakin Skywalker, a young slave strong with th...",6.583,136.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/1893-star-war...,0.245806
4,Star Wars: The Rise of Skywalker,Action Adventure Science Fiction,The surviving Resistance faces the First Order...,6.248,142.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/181812-star-w...,0.240254
5,Star Wars: Episode III - Revenge of the Sith,Action Adventure Science Fiction,When the sinister Sith unveil a thousand-year-...,7.475,140.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/1895-star-war...,0.224306
6,Star Wars: The Clone Wars,Action Adventure Animation Science Fiction,"As the Clone Wars sweep through the galaxy, An...",6.163,98.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/12180-star-wa...,0.204137
7,Star Wars: The Last Jedi,Action Adventure Science Fiction,Rey develops her newly discovered abilities wi...,6.736,152.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/181808-star-w...,0.203513
8,Rebel Moon - Part One: A Child of Fire,Action Adventure Science Fiction,When the ruthless forces of the Motherworld th...,6.215,134.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/848326-rebel-...,0.200949
9,The Mandalorian and Grogu,Action Adventure Science Fiction,"The evil Empire has fallen, and Imperial warlo...",7.473,132.0,https://media.themoviedb.org/t/p/w220_and_h330...,https://www.themoviedb.org/movie/1228710-the-m...,0.185939


In [19]:
model_bundle = {
    "movies_ml": movies_ml,
    "tfidf_vectorizer": tfidf_vectorizer,
    "tfidf_matrix": tfidf_matrix,
    "knn_model": knn_model
}

joblib.dump(model_bundle, "movie_recommender.joblib", compress=3)

['movie_recommender.joblib']